# 06 · Double Q-learning i Dyna-Q

Dwa rozszerzenia sterowania z tabelą wartości akcji.

- **Double Q-learning** rozdziela wybór najlepszej akcji od jej oceny. Sprawdzisz wpływ szumu na maksimum z szacowanych wartości.
- **Dyna-Q** łączy aktualizacje po rzeczywistych przejściach z aktualizacjami uzyskanymi z modelu zbudowanego z obserwacji.
- Wystarczy Python 3. Uruchamiaj komórki od góry. Eksperymenty używają ustalonych ziaren losowania.

Wersja studencka. Uzupełnij dwa bloki TODO. Test i eksperyment każdej części uruchom ponownie po uzupełnieniu funkcji.

## 1. Double Q-learning · obciążenie przez maksimum

Agent zaczyna w stanie A. Po prawej kończy z nagrodą 0. Po lewej dostaje 0 i przechodzi do stanu B. Każda akcja w B kończy epizod. Nagroda w B to niezależne losowanie $+0{,}9$ lub $-1{,}1$, po $1/2$. **Oczekiwany zwrot w B wynosi $-0{,}1$ dla każdej akcji**. Przy $\gamma=1$ optymalna akcja w A to „w prawo”.

Rozważ dwie akcje w B i po jednym niezależnym pomiarze na akcję. Szansa, że przynajmniej jeden pomiar to $+0{,}9$, wynosi $3/4$. Średnia maksimum z pomiarów to $\tfrac34\cdot0{,}9+\tfrac14\cdot(-1{,}1)=0{,}4$. Drugi, niezależny pomiar wybranej akcji ma średnią $-0{,}1$.

W Double Q-learningu trzymamy $Q_1,Q_2$, a w każdym kroku losujemy tabelę aktualizowaną. Gdy aktualizujemy $Q_1$, wartości tej tabeli wybierają akcję $a^*$, a $Q_2$ ją ocenia:
$$
a^*=\arg\max_{a'} Q_1(s',a'),\qquad y=r+\gamma Q_2(s',a^*),\qquad
Q_1(s,a)\gets Q_1(s,a)+\alpha[y-Q_1(s,a)].
$$
Przy aktualizacji $Q_2$ zamieniamy role tabel. Po końcu epizodu cel to $y=r$. W środowisku wybieramy akcje metodą $\varepsilon$-zachłanną na podstawie średniej obu tabel. W działającym algorytmie tablice powstają z doświadczenia i mogą być zależne.

In [ ]:
from random import Random
from statistics import mean
from math import isclose

A, B = 0, 1
LEWO, PRAWO = 0, 1
KIERUNKI = (LEWO, PRAWO)


def wybierz_epsilon_zachlannie(akcje, ocena, epsilon, rng):
    if rng.random() < epsilon:
        return rng.choice(akcje)
    najlepsza = max(ocena(a) for a in akcje)
    remis = [a for a in akcje if ocena(a) == najlepsza]
    return rng.choice(remis)


def q_update(q, s, a, r, s_nowy, koniec, akcje_nowe, alfa, gamma):
    cel = r if koniec else r + gamma * max(q[s_nowy, b] for b in akcje_nowe)
    q[s, a] += alfa * (cel - q[s, a])

### Zadanie 1 · uzupełnij krok Double Q-learningu

Funkcja dostaje tabele już ustawione w rolach. Zmień tylko `aktualizowana[s,a]`.

1. Jeżeli epizod się kończy, cel wynosi $r$. `s_nowy` ma wtedy wartość `None`, a lista akcji jest pusta.
2. W pozostałych przypadkach wybierz najlepszą akcję w stanie `s_nowy` na podstawie `aktualizowana`. Wartość tej **samej akcji** odczytaj z `oceniajaca`.
3. Zastosuj krok uczenia $\alpha$. Sprawdź funkcję z danymi, dla których najlepsza akcja w każdej tabeli jest inna.

In [ ]:
def double_update(aktualizowana, oceniajaca, s, a, r, s_nowy, koniec,
                  akcje_nowe, alfa, gamma):

    # TODO 1: zbuduj cel dla końca i dla przejścia do kolejnego stanu.
    # TODO 2: zastosuj krok alfa w tabeli aktualizowanej.
    raise NotImplementedError("Uzupełnij double_update")

In [ ]:
def sprawdz_double():
    aktualizowana = {(A, LEWO): 0.4, (B, 0): 0.0, (B, 1): 0.2, (B, 2): 3.0}
    oceniajaca = {(A, LEWO): 4.0, (B, 0): 0.5, (B, 1): 5.0, (B, 2): -1.0}
    zapis = oceniajaca.copy()
    double_update(aktualizowana, oceniajaca, A, LEWO, 2.0, B, False, (0, 1, 2), 0.25, 0.5)
    assert isclose(aktualizowana[A, LEWO], 0.675)
    assert oceniajaca == zapis

    aktualizowana[B, 0] = 0.8
    double_update(aktualizowana, oceniajaca, B, 0, -1, None, True, (), 0.5, 0.99)
    assert isclose(aktualizowana[B, 0], -0.1)
    print("OK: wybór i ocena w różnych tabelach, terminal i izolacja tabeli.")


try:
    sprawdz_double()
except NotImplementedError as blad:
    double_gotowe = False
    print(blad)
else:
    double_gotowe = True

### Eksperyment · jak często wybieramy lewo w A?

Używamy dziesięciu akcji w B. Porównujemy algorytmy na 200 ziarnach. Każda para przebiegów ma ten sam wylosowany ciąg potencjalnych nagród w B, przypisanych do numerów epizodów. Nagroda z danego epizodu staje się obserwacją, gdy agent dotrze do B. Losowanie zachowania ma osobny generator.

Przewidź, gdzie udział wyborów lewa będzie większy. Przy rozpoznaniu przewagi prawej akcji i $\varepsilon=0{,}10$ sama eksploracja wybiera lewo w $5\%$ epizodów w średniej po wielu próbach.

In [ ]:
def udzial_lewa(nagrody_b, ziarno, podwojne, alfa=0.1, epsilon=0.1, k=10):
    rng_zachowanie = Random(10_000 + ziarno)
    rng_tabela = Random(20_000 + ziarno)
    akcje_b = tuple(range(k))
    q1 = {(s, a): 0.0 for s, akcje in ((A, KIERUNKI), (B, akcje_b)) for a in akcje}
    q2 = q1.copy()
    wybory_lewa = []

    def wybierz(s, akcje):
        def ocena(a):
            if podwojne:
                return (q1[s, a] + q2[s, a]) / 2
            return q1[s, a]
        return wybierz_epsilon_zachlannie(akcje, ocena, epsilon, rng_zachowanie)

    def aktualizuj(s, a, r, nastepny, koniec, akcje):
        if podwojne:
            celowa, druga = (q1, q2) if rng_tabela.random() < 0.5 else (q2, q1)
            double_update(celowa, druga, s, a, r, nastepny, koniec, akcje, alfa, 1.0)
        else:
            q_update(q1, s, a, r, nastepny, koniec, akcje, alfa, 1.0)

    for nagroda_b in nagrody_b:
        akcja = wybierz(A, KIERUNKI)
        wybory_lewa.append(akcja == LEWO)
        if akcja == PRAWO:
            aktualizuj(A, akcja, 0.0, None, True, ())
            continue
        aktualizuj(A, akcja, 0.0, B, False, akcje_b)
        akcja_b = wybierz(B, akcje_b)
        aktualizuj(B, akcja_b, nagroda_b, None, True, ())

    return tuple(wybory_lewa)


def sprawdz_maksimum(ziarna=range(200), epizody=200):
    pary = []
    for ziarno in ziarna:
        rng = Random(ziarno)
        nagrody = [rng.choice((-1.1, 0.9)) for _ in range(epizody)]
        zwykle = udzial_lewa(nagrody, ziarno, False)
        podwojne = udzial_lewa(nagrody, ziarno, True)
        pary.append((zwykle, podwojne))
    return pary


def pokaz_podsumowanie(pary):
    print("Epizody   Q-learning   Double Q   Różnica Q − Double Q")
    for poczatek in range(0, len(pary[0][0]), 25):
        koniec = min(poczatek + 25, len(pary[0][0]))
        q = mean(mean(zwykle[poczatek:koniec]) for zwykle, _ in pary)
        dq = mean(mean(podwojne[poczatek:koniec]) for _, podwojne in pary)
        print(f"{poczatek + 1:>3}–{koniec:<3}    {q:>9.1%}  {dq:>9.1%}   {100*(q-dq):>6.1f} p.p.")
    roznice = sorted(mean(z[-25:]) - mean(d[-25:]) for z, d in pary)
    dolny = len(roznice) // 10
    print(f"Środkowe 80% różnic między sparowanymi ziarnami na końcu: "
          f"{100 * roznice[dolny]:.0f} do {100 * roznice[-dolny-1]:.0f} p.p.")


if double_gotowe:
    pokaz_podsumowanie(sprawdz_maksimum())
else:
    print("Najpierw uzupełnij funkcję double_update i uruchom jej test.")

**Po eksperymencie.** Porównaj epizody 1–25 i 176–200. Podaj udział wyborów lewa z obu metod. W którym kroku zwykły Q-learning korzysta z maksimum zaszumionych wartości? Jak ten krok zmienia Double Q-learning?

## 2. Dyna-Q · uczenie i planowanie z doświadczenia

W korytarzu agent rusza z pola 0. Wejście na pole 4 daje $+1$ i kończy epizod. Wszystkie pozostałe ruchy dają 0. Na lewym brzegu komenda w lewo zatrzymuje agenta na polu 0.

Po rzeczywistej obserwacji $(s,a,r,s',\mathrm{koniec})$:

1. Wykonujemy jedną aktualizację Q-learningu.
2. Zapisujemy w modelu $M(s,a)=(r,s',\mathrm{koniec})$.
3. Wykonujemy $n$ aktualizacji planowania. Każdorazowo losujemy parę $(\tilde{s},\tilde{a})$ z zapisanych par i odczytujemy jej skutek z $M$. Używamy tej samej reguły Q-learningu:
$$
Q(s,a)\gets Q(s,a)+\alpha(y-Q(s,a)),\qquad
y=\begin{cases}
r & \text{gdy epizod się kończy},\\
r+\gamma\max_b Q(s',b) & \text{w przeciwnym razie}.
\end{cases}
$$

Korytarz ma przejścia deterministyczne, więc ostatnia obserwacja danej pary wystarczy do zapamiętania jej skutku. W środowisku losowym model może przechowywać częstości różnych zaobserwowanych skutków.

### Zadanie 2 · dodaj aktualizacje planowania

Zaimplementuj `dyna_step`. Funkcja dostaje jedną rzeczywistą obserwację, używa dostarczonego `q_update`, modyfikuje `model` i zwraca liczbę wykonanych aktualizacji modelowych.

1. Rozpakuj obserwację i wykonaj krok `q_update`. W terminalu podaj puste akcje następcy, a w pozostałych przypadkach `KIERUNKI`.
2. Zapisz `(r, s_nowy, koniec)` pod kluczem `(s,a)`.
3. `n_planowania` razy wylosuj klucz przez `rng.choice(tuple(model))`. Pobierz skutek z modelu i wykonaj kolejny krok `q_update`.

Test terminala sprawdza jedną aktualizację rzeczywistą oraz trzy modelowe. Wszystkie cztery przesuwają $Q$ w kierunku $1$ z krokiem $1/2$.

In [ ]:
def dyna_step(q, model, obserwacja, n_planowania, rng, alfa=0.3, gamma=0.9):

    # TODO 1: użyj obserwacji w q_update i zapamiętaj jej skutek w modelu.
    # TODO 2: wykonaj aktualizacje z modelu i zwróć ich liczbę.
    raise NotImplementedError("Uzupełnij dyna_step")

In [ ]:
def sprawdz_dyna():
    q = {(3, a): 0.0 for a in KIERUNKI}
    model = {}
    obserwacja = (3, PRAWO, 1.0, 4, True)
    plan = dyna_step(q, model, obserwacja, 3, Random(7), alfa=0.5)
    assert model == {(3, PRAWO): (1.0, 4, True)}
    assert plan == 3
    assert isclose(q[3, PRAWO], 0.9375)

    q = {(s, a): 0.0 for s in range(4) for a in KIERUNKI}
    q[3, PRAWO] = 1.0
    model = {}
    plan = dyna_step(q, model, (2, PRAWO, 0.0, 3, False), 0, Random(7), alfa=0.5, gamma=0.9)
    assert model == {(2, PRAWO): (0.0, 3, False)}
    assert plan == 0
    assert isclose(q[2, PRAWO], 0.45)
    print("OK: pamięć modelu, terminal, przejście ze stanem następnym i licznik aktualizacji.")


try:
    sprawdz_dyna()
except NotImplementedError as blad:
    dyna_gotowe = False
    print(blad)
else:
    dyna_gotowe = True

### Eksperyment · te same przejścia, różna liczba aktualizacji modelowych

Dostarczamy wariantom ten sam zapis pięciu epizodów, w których agent idzie korytarzem do celu. Dzięki temu liczba rzeczywistych przejść i kolejność obserwacji są identyczne. Różni się praca wykonana na modelu, więc porównujemy wykorzystanie doświadczenia.

Przewidź, który wariant rozpozna wartość ruchu w prawo z pola 0 po pierwszych czterech rzeczywistych przejściach. Punktem odniesienia jest $Q^*(0,\mathrm{prawo})=0{,}9^3=0{,}729$. Do celu prowadzą cztery kroki, a tylko ostatni daje nagrodę.

In [ ]:
def przejdz_w_prawo():
    return [(s, PRAWO, float(s == 3), s + 1, s == 3) for s in range(4)]


def ucz_z_zapisu(zapis, n_planowania, ziarno):
    rng = Random(ziarno)
    q = {(s, a): 0.0 for s in range(4) for a in KIERUNKI}
    model = {}
    plan = 0
    wyniki = []
    for i, obserwacja in enumerate(zapis, start=1):
        plan += dyna_step(q, model, obserwacja, n_planowania, rng)
        wyniki.append((i, plan, q[0, PRAWO]))
    return wyniki


def pokaz_dyna():
    zapis = przejdz_w_prawo() * 5
    print("Plan na krok     Realne przejścia     Modelowe aktualizacje     Średnie Q(0,prawo)")
    for n in (0, 5, 20):
        proby = [ucz_z_zapisu(zapis, n, ziarno) for ziarno in range(30)]
        for i in (4, 8, 12, 20):
            realne, plan, _ = proby[0][i - 1]
            ocena = mean(proba[i - 1][2] for proba in proby)
            print(f"{n:>5}            {realne:>12}         {plan:>20}          {ocena:>13.4f}")


if dyna_gotowe:
    pokaz_dyna()
else:
    print("Najpierw uzupełnij funkcję dyna_step i uruchom jej test.")

## Wniosek

- Które wyniki pokazują wpływ oszacowań z szumem na decyzję w A?
- Ile rzeczywistych przejść i ile modelowych aktualizacji wykonuje wariant z 5 krokami planowania po 12 obserwacjach?
- Czy pomiar $Q(0,\mathrm{prawo})$ po dostarczeniu zapisu pozwala stwierdzić, jak często agent sam odkryłby cel? Uzasadnij.
- Co zmieniłby model, który zapamiętał jeden pechowy wynik i uznał go za deterministyczny?

**Kolejny temat.** W większych środowiskach aproksymator pozwala przewidywać wartości w podobnych stanach. DQN wykorzystuje gradient straty dotyczącej $Q$. Metody gradientu polityki zmieniają bezpośrednio parametry reguły wyboru akcji.